# LABORATORIO N.° 03 — La métrica que importa

**Curso:** Analítica Empresarial Integrada  
**Semana 3:** KPI accionables, North Star Metric y árbol de métricas  
**Docente:** Pilar Rocío Sayán Mejía  
**Periodo:** 2026-II

**Fuente real:** UCI Machine Learning Repository — Online Retail (ID 352).

> Objetivo del notebook: conectar **objetivo → North Star → drivers → guardrails → KPI → tablero → interpretación → decisión** usando transacciones reales.
---

**Equipo N.°:** _____ &nbsp;&nbsp;&nbsp; **Sección:** 6C28 &nbsp;&nbsp;&nbsp; **Fecha:** 02/09/2026

**Apellidos y nombres del estudiante:** _Furushio Casanave, Piero Hideki_

**Integrantes del equipo:** _(apellidos y nombres de todos los integrantes)_

---


## Agenda de laboratorio — 7:00 p. m. a 10:10 p. m.

**Duración total:** 190 minutos · **Receso:** 15 minutos · **Trabajo efectivo:** 175 minutos.

| Horario | Tiempo | Desarrollo |
|---|---:|---|
| 7:00–7:10 | 10 min | Apertura del caso y presentación del problema de medición. |
| 7:10–7:35 | 25 min | Actividad 1. Revisión de conceptos: del dato a la decisión. |
| 7:35–8:00 | 25 min | Actividad 2. Descarga desde UCI, auditoría y limpieza documentada. |
| 8:00–8:30 | 30 min | Actividad 2. Periodo comparable, recurrencia y North Star. Reto 1. |
| 8:30–8:45 | 15 min | **RECESO** |
| 8:45–9:15 | 30 min | Actividad 2. Árbol de métricas, guardrails, Polars y DuckDB. |
| 9:15–9:35 | 20 min | Actividad 2. Tablero de decisión en Plotly. Reto 2. |
| 9:35–10:00 | 25 min | **Reto de aplicación y retroalimentación.** Ejercicios 1 a 5. |
| 10:00–10:10 | 10 min | Diccionario de KPI, informe ejecutivo y ticket de salida. |

> **Regla de trabajo:** no avance de bloque sin registrar la interpretación solicitada. El objetivo no es ejecutar celdas, sino convertir datos en evidencia para una decisión.


## Actividad 1 — Revisión de conceptos: del dato a la decisión (25 minutos)

**Propósito.** Establecer con precisión el vocabulario de medición antes de programar. La confusión entre dato, métrica, indicador y KPI constituye la causa más frecuente de tableros que no sustentan ninguna decisión.

**Instrucciones.** Complete la tabla con definiciones elaboradas con sus propias palabras. No se admite la reproducción literal de fuentes externas ni de sistemas generativos. La columna de la derecha contiene una pregunta de apoyo: si su definición permite responderla, la definición es suficiente; si no lo permite, corríjala antes de continuar.

**Evidencia esperada.** Tabla completa con las definiciones registradas.

| Concepto | Definición elaborada por el estudiante | Pregunta de apoyo |
|---|---|---|
| Dato | Valor bruto sin procesar, sin contexto ni interpretación.| ¿En qué se diferencia un dato de una métrica? Proponga un ejemplo de la base utilizada. |
| Métrica | 	Cálculo cuantitativo a partir de datos brutos. | ¿Toda métrica calculada correctamente resulta útil para decidir? Justifique. |
| Indicador | Métrica con contexto, umbral o referencia que permite evaluar desempeño. | ¿Qué debe añadirse a una métrica para que constituya un indicador? |
| KPI | Indicador crítico ligado directamente a un objetivo estratégico clave. | ¿Por qué una organización no puede sostener veinte KPI simultáneos? |
| Meta | Valor objetivo deseado para un indicador en un periodo determinado. | ¿Qué distingue un valor observado en la base de una meta propuesta para el ejercicio? |
| North Star | Métrica única que resume el valor entregado al cliente y guía el crecimiento. | ¿Qué condición debe cumplir una North Star para no convertirse en métrica de vanidad? |
| Driver | Variable que influye directamente en la North Star. | ¿Cómo se comprueba que una variable es efectivamente driver de la North Star? |
| Guardrail | Indicador de control que previene efectos secundarios negativos al optimizar la North Star. | ¿Qué ocurre si una organización optimiza su North Star sin vigilar los guardrails? |

**Criterio de cierre.** No se avanza al desarrollo práctico mientras existan conceptos sin definición registrada.


## Actividad 2 — Desarrollo práctico y ejecución

### Presentación del caso

Una empresa minorista en línea del Reino Unido cuenta con un registro histórico de transacciones que incluye facturas, productos, cantidades, fechas, precios unitarios, clientes y países. La dirección necesita transformar estos registros en un sistema de métricas que permita distinguir crecimiento útil de crecimiento aparente. Para ello, no basta con observar ventas acumuladas o cantidad de clientes: es necesario identificar qué indicadores representan valor recurrente para el cliente y cuáles pueden orientar una decisión empresarial.

Durante el laboratorio, el equipo trabajará con el conjunto Online Retail del UCI Machine Learning Repository. A partir de los datos reales, deberá auditar y preparar las transacciones, identificar clientes recurrentes, proponer y justificar una North Star, descomponerla en drivers y guardrails, construir un diccionario de KPI y elaborar un tablero de decisión en Plotly. El análisis deberá terminar con hallazgos cuantitativos y una acción empresarial concreta, diferenciando en todo momento los valores observados en la base de las metas o umbrales académicos propuestos para el ejercicio.


## Paso 1 — Preparación reproducible del entorno


In [1]:
%pip install -q ucimlrepo==0.0.7 polars==1.17.1 duckdb==1.1.3

import polars as pl
import duckdb
import pandas as pd            # solo para recibir la descarga de UCI y alimentar Plotly Express
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from ucimlrepo import fetch_ucirepo

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(160)

# Formato REAL de las fechas de esta fuente: 12/1/2010 8:26 -> mes/dia/anio hora:min.
# Se declara de forma explicita en lugar de dejar que la libreria lo adivine.
FORMATO_FECHA = "%m/%d/%Y %H:%M"

print("polars:", pl.__version__, "| duckdb:", duckdb.__version__)
print("Entorno listo.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 49.4 MB/s eta 0:00:00
polars: 1.17.1 | duckdb: 1.1.3
Entorno listo.


## Paso 2 — Descarga de datos reales desde UCI

El conjunto **Online Retail** contiene transacciones de una empresa minorista en línea registrada en el Reino Unido entre diciembre de 2010 y diciembre de 2011. Los códigos de factura que empiezan con `C` representan cancelaciones. No se generan registros artificiales.


In [2]:
online_retail = fetch_ucirepo(id=352)
df = pl.from_pandas(online_retail.data.original)   # la descarga llega en pandas; se pasa a Polars

print("Dataset:", online_retail.metadata.get("name"))
print("Filas y columnas:", df.shape)
print("Columnas:", df.columns)
display(df.head())


Dataset: Online Retail
Filas y columnas: (541909, 8)
Columnas: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
str,str,str,i64,str,f64,f64,str
"""536365""","""85123A""","""WHITE HANGING HEART T-LIGHT HO…",6,"""12/1/2010 8:26""",2.55,17850.0,"""United Kingdom"""
"""536365""","""71053""","""WHITE METAL LANTERN""",6,"""12/1/2010 8:26""",3.39,17850.0,"""United Kingdom"""
"""536365""","""84406B""","""CREAM CUPID HEARTS COAT HANGER""",8,"""12/1/2010 8:26""",2.75,17850.0,"""United Kingdom"""
"""536365""","""84029G""","""KNITTED UNION FLAG HOT WATER B…",6,"""12/1/2010 8:26""",3.39,17850.0,"""United Kingdom"""
"""536365""","""84029E""","""RED WOOLLY HOTTIE WHITE HEART.""",6,"""12/1/2010 8:26""",3.39,17850.0,"""United Kingdom"""


### Control de trazabilidad
Registre: número de filas, columnas, rango de fechas y países presentes.

**Respuesta:**
El dataset contiene 541 909 filas y 8 columnas (InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country). El rango de fechas va del 2010-12-01 08:26 al 2011-12-09 12:50. Se registran 38 países distintos, con Reino Unido concentrando la mayor parte de las transacciones.


## Paso 3 — Auditoría inicial y limpieza documentada


In [3]:
df = df.with_columns([
    pl.col("InvoiceNo").cast(pl.Utf8),
    pl.col("CustomerID").cast(pl.Utf8),
    pl.col("Quantity").cast(pl.Float64, strict=False),
    pl.col("UnitPrice").cast(pl.Float64, strict=False),
    pl.col("InvoiceDate").cast(pl.Utf8)
      .str.to_datetime(format=FORMATO_FECHA, strict=False).alias("InvoiceDate"),
])
df = df.with_columns([
    pl.col("InvoiceNo").str.to_uppercase().str.starts_with("C").alias("es_cancelacion"),
    (pl.col("Quantity") * pl.col("UnitPrice")).alias("importe_linea"),
])

# Control de parseo: si el formato declarado no fuese el correcto, aqui apareceria
# un conteo de fechas nulas. pandas habria "adivinado" un formato sin avisar;
# Polars obliga a declararlo y deja el error a la vista.
nulas = df["InvoiceDate"].null_count()
print("Fechas que no pudieron parsearse:", nulas)
if nulas:
    raise ValueError("El formato declarado en FORMATO_FECHA no corresponde a la fuente.")

# Control de integridad de la fuente. Si UCI republica el archivo, el laboratorio
# se detiene aqui en vez de producir numeros que no coinciden con el solucionario.
FILAS_ESPERADAS = 541_909
FACTURAS_ESPERADAS = 25_900

if df.height != FILAS_ESPERADAS:
    raise ValueError(
        f"La fuente cambio: se esperaban {FILAS_ESPERADAS:,} filas y llegaron {df.height:,}. "
        "Avise al docente antes de continuar."
    )
if df["InvoiceNo"].n_unique() != FACTURAS_ESPERADAS:
    raise ValueError(
        f"La fuente cambio: se esperaban {FACTURAS_ESPERADAS:,} facturas distintas "
        f"y llegaron {df['InvoiceNo'].n_unique():,}."
    )
print("Integridad verificada:", f"{df.height:,}", "filas y",
      f"{df['InvoiceNo'].n_unique():,}", "facturas, como se esperaba.")

control = pl.DataFrame({
    "indicador": ["filas", "facturas", "clientes", "cancelaciones",
                  "cantidades_no_positivas", "precios_no_positivos"],
    "valor": [df.height,
              df["InvoiceNo"].n_unique(),
              df["CustomerID"].drop_nulls().n_unique(),
              int(df["es_cancelacion"].sum()),
              int((df["Quantity"] <= 0).sum()),
              int((df["UnitPrice"] <= 0).sum())],
})
display(control)
print("Rango:", df["InvoiceDate"].min(), "->", df["InvoiceDate"].max())


Fechas que no pudieron parsearse: 0
Integridad verificada: 541,909 filas y 25,900 facturas, como se esperaba.


indicador,valor
str,i64
"""filas""",541909
"""facturas""",25900
"""clientes""",4372
"""cancelaciones""",9288
"""cantidades_no_positivas""",10624
"""precios_no_positivos""",2517


Rango: 2010-12-01 08:26:00 -> 2011-12-09 12:50:00


In [4]:
compras = df.filter(
    (~pl.col("es_cancelacion"))
    & (pl.col("Quantity") > 0)
    & (pl.col("UnitPrice") > 0)
    & pl.col("InvoiceDate").is_not_null()
    & pl.col("CustomerID").is_not_null()
)

print("Filas originales:", df.height)
print("Filas de compra validas:", compras.height)
print("Proporcion conservada: {:.1%}".format(compras.height / df.height))


Filas originales: 541909
Filas de compra validas: 397884
Proporcion conservada: 73.4%


### Reto de calidad
1. ¿Qué sesgo aparece si contamos cancelaciones como ventas?  
2. ¿Por qué no debemos borrar esos registros de la fuente original?

**Respuesta:**
1. Contar las cancelaciones como ventas sobrestima tanto los ingresos como la cantidad de compras: una factura anulada (código con prefijo `C`) no representa valor entregado ni ingreso definitivo, así que incluirla infla artificialmente el desempeño comercial.
2. No deben borrarse de la fuente original porque son evidencia operativa real: permiten auditar devoluciones y calcular la tasa de cancelación como guardrail. El filtro se aplica solo sobre la copia de trabajo (`compras`), preservando `df` intacto para trazabilidad.


## Paso 4 — Definición del periodo comparable

La fuente empieza el 1/12/2010 y termina el 9/12/2011. Para no comparar meses incompletos con meses completos, el laboratorio informa sobre **enero–noviembre de 2011**.

**Cuidado con el sesgo de ventana.** La tabla de facturas se construye sobre *todo* el historial disponible, y el recorte a la ventana se aplica **después** de determinar la recurrencia. Si se recorta antes, un cliente que compró en diciembre de 2010 y volvió en enero de 2011 aparece como comprador nuevo, y la North Star crece de forma artificial en los primeros meses: en esta base, enero pasa de 570 a 246 compras recurrentes, un 57 % menos, solo por haber filtrado en el orden equivocado.


In [5]:
compras = compras.with_columns(pl.col("InvoiceDate").dt.strftime("%Y-%m").alias("mes"))
INICIO, FIN = pl.datetime(2011, 1, 1), pl.datetime(2011, 12, 1)

# La tabla se construye sobre TODO el historial: el recorte a la ventana
# comparable se hace mas adelante, una vez determinada la recurrencia.
facturas = (compras.group_by(["InvoiceNo", "CustomerID", "mes"])
            .agg([pl.col("InvoiceDate").min().alias("fecha_factura"),
                  pl.col("importe_linea").sum().alias("importe_factura"),
                  pl.col("Quantity").sum().alias("unidades"),
                  pl.col("StockCode").count().alias("lineas")]))

en_ventana = facturas.filter((pl.col("fecha_factura") >= INICIO) & (pl.col("fecha_factura") < FIN))
print("Facturas en el historial completo :", facturas["InvoiceNo"].n_unique())
print("Facturas en la ventana comparable :", en_ventana["InvoiceNo"].n_unique())
print("Clientes en la ventana            :", en_ventana["CustomerID"].n_unique())
print("Ingresos en la ventana (GBP)      :", round(en_ventana["importe_factura"].sum(), 2))
display(facturas.head())


Facturas en el historial completo : 18532
Facturas en la ventana comparable : 16354
Clientes en la ventana            : 4173
Ingresos en la ventana (GBP)      : 7820501.22


InvoiceNo,CustomerID,mes,fecha_factura,importe_factura,unidades,lineas
str,str,str,datetime[μs],f64,f64,u32
"""561669""","""12507.0""","""2011-07""",2011-07-28 17:09:00,811.9,664.0,7
"""567509""","""14298.0""","""2011-09""",2011-09-20 14:51:00,752.42,730.0,29
"""568652""","""15841.0""","""2011-09""",2011-09-28 12:15:00,307.32,228.0,8
"""574697""","""16488.0""","""2011-11""",2011-11-06 13:42:00,144.23,109.0,7
"""573155""","""13632.0""","""2011-10""",2011-10-28 08:34:00,854.53,433.0,44


## Paso 5 — Recurrencia y North Star

**Regla operativa del laboratorio:** un cliente se considera recurrente desde su segunda factura válida, contada sobre **todo el historial disponible**. Su primera compra no se reclasifica retrospectivamente.

Enero de 2011 conserva un sesgo residual, porque solo dispone de un mes previo de historial. Por eso se marca como **mes de calentamiento** y se excluye de las comparaciones de variación mensual.


In [6]:
facturas = facturas.sort(["CustomerID", "fecha_factura", "InvoiceNo"])
facturas = facturas.with_columns(
    (pl.int_range(pl.len()).over("CustomerID") + 1).alias("n_compra_cliente"))

# Una factura es recurrente a partir de la SEGUNDA compra del cliente, en orden
# cronologico. La marca se calcula sobre el historial completo para no clasificar
# retroactivamente como recurrente la primera compra de un cliente que volvio despues.
facturas = facturas.with_columns(
    (pl.col("n_compra_cliente") >= 2).alias("es_recurrente"))

# Recien ahora se recorta a la ventana comparable: la recurrencia ya quedo
# determinada usando todo el historial, sin sesgo de ventana.
facturas = facturas.filter((pl.col("fecha_factura") >= INICIO) & (pl.col("fecha_factura") < FIN))

# Enero solo tiene un mes previo de historial: no es comparable en variacion.
MES_CALENTAMIENTO = "2011-01"

mensual = (facturas.group_by("mes")
           .agg([pl.col("InvoiceNo").n_unique().alias("facturas_validas"),
                 pl.col("CustomerID").n_unique().alias("clientes_activos"),
                 pl.col("importe_factura").sum().alias("ingresos")]))

rec = (facturas.filter("es_recurrente").group_by("mes")
       .agg([pl.col("InvoiceNo").n_unique().alias("compras_recurrentes"),
             pl.col("CustomerID").n_unique().alias("clientes_recurrentes"),
             pl.col("importe_factura").sum().alias("ingresos_recurrentes")]))

kpi = mensual.join(rec, on="mes", how="left").fill_null(0).sort("mes")
kpi = kpi.with_columns([
    (pl.col("compras_recurrentes") / pl.col("clientes_recurrentes")).alias("frecuencia_recurrente"),
    (100 * pl.col("ingresos_recurrentes") / pl.col("ingresos")).alias("participacion_ingreso_recurrente_pct"),
])
display(kpi)


mes,facturas_validas,clientes_activos,ingresos,compras_recurrentes,clientes_recurrentes,ingresos_recurrentes,frecuencia_recurrente,participacion_ingreso_recurrente_pct
str,u32,u32,f64,u32,u32,f64,f64,f64
"""2011-01""",987,741,569445.04,570,370,296614.01,1.540541,52.088259
"""2011-02""",997,758,447137.35,617,412,299937.36,1.497573,67.079469
"""2011-03""",1321,974,595500.76,869,563,411030.13,1.543517,69.022604
"""2011-04""",1149,856,469200.361,849,587,357273.97,1.446337,76.145289
"""2011-05""",1555,1056,678594.56,1271,809,566039.71,1.571075,83.413535
"""2011-06""",1393,991,661213.69,1151,773,570140.0,1.489004,86.226285
"""2011-07""",1331,949,600091.011,1143,781,532074.68,1.463508,88.665664
"""2011-08""",1280,935,645343.9,1111,778,568263.68,1.428021,88.055947
"""2011-09""",1755,1266,952838.382,1456,996,806684.471,1.461847,84.661207


### North Star propuesta
**Compras válidas de clientes recurrentes por mes.**

Justifique con los cuatro criterios: valor para el cliente, vínculo con valor empresarial, capacidad de influencia del equipo y descomposición en drivers.

**Respuesta:**
- **Valor para el cliente:** una compra recurrente confirma que el cliente ya validó la propuesta de valor y decide volver, algo que una venta única no garantiza.
- **Vínculo con valor empresarial:** la participación del ingreso recurrente crece de 52 % (enero) a 90 % (noviembre), es decir, este segmento explica la mayor parte del ingreso mensual.
- **Capacidad de influencia del equipo:** retención, marketing y servicio al cliente pueden actuar directamente sobre la recurrencia (campañas de reactivación, fidelización), a diferencia de factores externos.
- **Descomposición en drivers:** la North Star se factoriza de forma exacta en clientes recurrentes × frecuencia recurrente (error de reconstrucción prácticamente 0), lo que permite diagnosticar si un cambio proviene de más clientes o de mayor frecuencia.


### Reto 1 — ¿Vanidad o acción?
Clasifique: productos totales, clientes activos mensuales, compras recurrentes, ingresos acumulados, tasa de cancelación y países con ventas. Para cada una indique qué decisión permite tomar.

**Respuesta:**
- **Productos totales:** vanidad; describe el catálogo, no el comportamiento del cliente, y no orienta ninguna decisión inmediata.
- **Clientes activos mensuales:** accionable con matices; útil para dimensionar capacidad operativa, pero al no distinguir nuevos de recurrentes no dice si el negocio retiene valor.
- **Compras recurrentes:** accionable; es la North Star y permite decidir dónde invertir en retención.
- **Ingresos acumulados:** vanidad en términos absolutos (crece casi por definición); solo es útil si se compara contra una meta o periodo comparable.
- **Tasa de cancelación:** accionable; es un guardrail que dispara revisión de calidad de producto o de despacho cuando sube.
- **Países con ventas:** vanidad; comunica alcance geográfico pero no orienta ninguna acción concreta de gestión.


# ☕ RECESO — 8:30 p. m. a 8:45 p. m.

## Paso 6 — Árbol de métricas

El árbol es una **hipótesis de gestión**, no una demostración causal.

**Objetivo → North Star → drivers → guardrails**

- Objetivo: incrementar valor recurrente sin deteriorar calidad.
- North Star: compras válidas de clientes recurrentes / mes.
- Driver 1: clientes recurrentes activos.
- Driver 2: frecuencia de compra por recurrente.
- Guardrail 1: tasa de cancelación.
- Guardrail 2: concentración de ingresos en Top 10 clientes.


In [7]:
# Validacion aritmetica del primer nivel del arbol
kpi = kpi.with_columns(
    (pl.col("clientes_recurrentes") * pl.col("frecuencia_recurrente")).alias("ns_reconstruida"))
kpi = kpi.with_columns(
    (pl.col("compras_recurrentes") - pl.col("ns_reconstruida")).alias("error_reconstruccion"))
display(kpi.select(["mes", "compras_recurrentes", "clientes_recurrentes",
                    "frecuencia_recurrente", "error_reconstruccion"]))


mes,compras_recurrentes,clientes_recurrentes,frecuencia_recurrente,error_reconstruccion
str,u32,u32,f64,f64
"""2011-01""",570,370,1.540541,0.0
"""2011-02""",617,412,1.497573,0.0
"""2011-03""",869,563,1.543517,0.0
"""2011-04""",849,587,1.446337,1.1369e-13
"""2011-05""",1271,809,1.571075,0.0
"""2011-06""",1151,773,1.489004,0.0
"""2011-07""",1143,781,1.463508,0.0
"""2011-08""",1111,778,1.428021,0.0
"""2011-09""",1456,996,1.461847,0.0


## Paso 7 — Guardrail 1: tasa de cancelación


In [8]:
base_total = (df.filter(pl.col("InvoiceDate").is_not_null()
                        & pl.col("CustomerID").is_not_null()
                        & (pl.col("InvoiceDate") >= INICIO)
                        & (pl.col("InvoiceDate") < FIN))
                .with_columns(pl.col("InvoiceDate").dt.strftime("%Y-%m").alias("mes")))

inv_total = base_total.select(["mes", "InvoiceNo", "CustomerID", "es_cancelacion"]).unique()
cancel = (inv_total.group_by("mes")
          .agg([pl.col("InvoiceNo").n_unique().alias("facturas_total"),
                pl.col("es_cancelacion").sum().alias("facturas_canceladas")])
          .with_columns((100 * pl.col("facturas_canceladas") / pl.col("facturas_total"))
                        .alias("tasa_cancelacion_pct"))
          .sort("mes"))
display(cancel)


mes,facturas_total,facturas_canceladas,tasa_cancelacion_pct
str,u32,u32,f64
"""2011-01""",1236,249,20.145631
"""2011-02""",1202,204,16.971714
"""2011-03""",1619,298,18.406424
"""2011-04""",1384,235,16.979769
"""2011-05""",1849,294,15.900487
"""2011-06""",1707,314,18.394845
"""2011-07""",1593,262,16.446955
"""2011-08""",1544,263,17.033679
"""2011-09""",2078,322,15.495669


## Paso 8 — Guardrail 2: concentración Top 10 clientes


In [9]:
cliente_mes = (facturas.group_by(["mes", "CustomerID"])
               .agg(pl.col("importe_factura").sum().alias("ingreso_cliente")))

ordenado = cliente_mes.sort(["mes", "ingreso_cliente"], descending=[False, True])
ordenado = ordenado.with_columns((pl.int_range(pl.len()).over("mes") + 1).alias("ranking"))

conc = (ordenado.group_by("mes")
        .agg([pl.col("ingreso_cliente").sum().alias("ingreso_mes"),
              pl.col("ingreso_cliente").filter(pl.col("ranking") <= 10)
                .sum().alias("ingreso_top10")])
        .with_columns((100 * pl.col("ingreso_top10") / pl.col("ingreso_mes"))
                      .alias("concentracion_top10_pct"))
        .sort("mes"))

display(conc)


mes,ingreso_mes,ingreso_top10,concentracion_top10_pct
str,f64,f64,f64
"""2011-01""",569445.04,193292.89,33.944082
"""2011-02""",447137.35,89670.06,20.054254
"""2011-03""",595500.76,113585.39,19.073929
"""2011-04""",469200.361,73375.7,15.638458
"""2011-05""",678594.56,128119.53,18.880129
"""2011-06""",661213.69,193184.16,29.2166
"""2011-07""",600091.011,121223.38,20.200833
"""2011-08""",645343.9,160995.32,24.947213
"""2011-09""",952838.382,233643.58,24.520799


In [10]:
kpi = (kpi
       .join(cancel.select(["mes", "tasa_cancelacion_pct"]), on="mes", how="left")
       .join(conc.select(["mes", "concentracion_top10_pct"]), on="mes", how="left")
       .sort("mes"))

# Variacion mensual de la North Star y de sus drivers
kpi = kpi.with_columns([
    (100 * (pl.col("compras_recurrentes") / pl.col("compras_recurrentes").shift(1) - 1)).alias("var_ns_pct"),
    (100 * (pl.col("clientes_recurrentes") / pl.col("clientes_recurrentes").shift(1) - 1)).alias("var_clientes_rec_pct"),
    (100 * (pl.col("frecuencia_recurrente") / pl.col("frecuencia_recurrente").shift(1) - 1)).alias("var_frecuencia_pct"),
])
display(kpi)


mes,facturas_validas,clientes_activos,ingresos,compras_recurrentes,clientes_recurrentes,ingresos_recurrentes,frecuencia_recurrente,participacion_ingreso_recurrente_pct,ns_reconstruida,error_reconstruccion,tasa_cancelacion_pct,concentracion_top10_pct,var_ns_pct,var_clientes_rec_pct,var_frecuencia_pct
str,u32,u32,f64,u32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2011-01""",987,741,569445.04,570,370,296614.01,1.540541,52.088259,570.0,0.0,20.145631,33.944082,null,null,null
"""2011-02""",997,758,447137.35,617,412,299937.36,1.497573,67.079469,617.0,0.0,16.971714,20.054254,8.245614,11.351351,-2.789133
"""2011-03""",1321,974,595500.76,869,563,411030.13,1.543517,69.022604,869.0,0.0,18.406424,19.073929,40.842788,36.650485,3.067901
"""2011-04""",1149,856,469200.361,849,587,357273.97,1.446337,76.145289,849.0,1.1369e-13,16.979769,15.638458,-2.301496,4.262877,-6.295983
"""2011-05""",1555,1056,678594.56,1271,809,566039.71,1.571075,83.413535,1271.0,0.0,15.900487,18.880129,49.705536,37.819421,8.624412
"""2011-06""",1393,991,661213.69,1151,773,570140.0,1.489004,86.226285,1151.0,0.0,18.394845,29.2166,-9.441385,-4.449938,-5.223907
"""2011-07""",1331,949,600091.011,1143,781,532074.68,1.463508,88.665664,1143.0,0.0,16.446955,20.200833,-0.695048,1.034929,-1.712256
"""2011-08""",1280,935,645343.9,1111,778,568263.68,1.428021,88.055947,1111.0,0.0,17.033679,24.947213,-2.79965,-0.384123,-2.424841
"""2011-09""",1755,1266,952838.382,1456,996,806684.471,1.461847,84.661207,1456.0,0.0,15.495669,24.520799,31.053105,28.020566,2.368791


### Pregunta 2
Identifique el mes con mayor tasa de cancelación y el mes con mayor concentración Top 10. ¿Qué riesgo representa cada uno?

**Respuesta:**
Enero de 2011 registra tanto la mayor tasa de cancelación (20,1 %) como la mayor concentración Top 10 (33,9 %). Una tasa de cancelación alta expone fricción en el proceso de compra o problemas de calidad que erosionan el ingreso válido. Una concentración Top 10 alta expone dependencia: perder pocos clientes grandes comprometería una porción desproporcionada del ingreso mensual.


## Paso 9 — Comparación reproducible con Polars y DuckDB


In [11]:
ranking_caida = (kpi
                 .select(["mes", "compras_recurrentes", "var_ns_pct",
                          "tasa_cancelacion_pct", "concentracion_top10_pct"])
                 .sort("var_ns_pct"))
ranking_caida


mes,compras_recurrentes,var_ns_pct,tasa_cancelacion_pct,concentracion_top10_pct
str,u32,f64,f64,f64
"""2011-01""",570,null,20.145631,33.944082
"""2011-06""",1151,-9.441385,18.394845,29.2166
"""2011-08""",1111,-2.79965,17.033679,24.947213
"""2011-04""",849,-2.301496,16.979769,15.638458
"""2011-07""",1143,-0.695048,16.446955,20.200833
"""2011-10""",1571,7.898352,14.759169,21.768432
"""2011-02""",617,8.245614,16.971714,20.054254
"""2011-09""",1456,31.053105,15.495669,24.520799
"""2011-03""",869,40.842788,18.406424,19.073929


In [12]:
# DuckDB consulta el DataFrame de Polars directamente, sin copias intermedias.
consulta = duckdb.sql("""
SELECT mes,
       compras_recurrentes,
       ROUND(var_ns_pct, 1) AS var_ns_pct,
       ROUND(tasa_cancelacion_pct, 1) AS cancelacion_pct,
       ROUND(concentracion_top10_pct, 1) AS concentracion_top10_pct
FROM kpi
ORDER BY var_ns_pct ASC NULLS LAST
""").pl()
display(consulta)


mes,compras_recurrentes,var_ns_pct,cancelacion_pct,concentracion_top10_pct
str,u32,f64,f64,f64
"""2011-06""",1151,-9.4,18.4,29.2
"""2011-08""",1111,-2.8,17.0,24.9
"""2011-04""",849,-2.3,17.0,15.6
"""2011-07""",1143,-0.7,16.4,20.2
"""2011-10""",1571,7.9,14.8,21.8
"""2011-02""",617,8.2,17.0,20.1
"""2011-09""",1456,31.1,15.5,24.5
"""2011-03""",869,40.8,18.4,19.1
"""2011-11""",2334,48.6,13.9,15.7


## Paso 10 — Tablero de decisión


In [13]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=kpi["mes"].to_list(), y=kpi["compras_recurrentes"].to_list(),
                         mode="lines+markers", name="North Star"))

# El tablero incorpora el driver principal junto al resultado: un tablero que
# muestra la North Star sin su driver no permite decidir donde actuar.
fig.add_trace(go.Scatter(x=kpi["mes"].to_list(), y=kpi["clientes_recurrentes"].to_list(),
                         mode="lines+markers", name="Clientes recurrentes (driver)"))

fig.update_layout(title="North Star y driver principal - 2011", xaxis_title="Mes",
                  yaxis_title="Cantidad", hovermode="x unified")
fig.show()


In [14]:
fig2 = px.line(kpi.to_pandas(), x="mes", y=["tasa_cancelacion_pct", "concentracion_top10_pct"],
               markers=True, title="Guardrails observados - cancelacion y concentracion")
fig2.update_layout(yaxis_title="Porcentaje (%)", xaxis_title="Mes", legend_title_text="Guardrail")
fig2.show()


In [15]:
fig3 = px.line(kpi.to_pandas(), x="mes", y=["participacion_ingreso_recurrente_pct"],
               markers=True, title="Participacion del ingreso proveniente de compras recurrentes")
fig3.update_layout(yaxis_title="Porcentaje (%)", xaxis_title="Mes")
fig3.show()


### Interpretación obligatoria
No basta con decir “la línea bajó”. Responda:
1. ¿Cuánto cambió y en qué periodo?
2. ¿Cómo se comportó el driver principal?
3. ¿Algún guardrail empeoró al mismo tiempo?
4. ¿Qué puede afirmarse como evidencia y qué es solo una inferencia?
5. ¿Quién debería decidir y qué acción ejecutaría?

**Respuesta:**
1. La participación del ingreso recurrente subió de 52,1 % (enero) a 90,4 % (noviembre), casi 38 puntos porcentuales en once meses, con una caída puntual entre enero y febrero.
2. El driver principal (clientes recurrentes) acompañó la tendencia: creció de 370 a 1397 en el mismo periodo.
3. Ningún guardrail empeoró de forma sostenida: la tasa de cancelación bajó de 20,1 % a 13,9 % y la concentración Top 10 bajó de 33,9 % a 15,7 %.
4. Evidencia: los porcentajes calculados directamente de las facturas. Inferencia: que la mejora se debe a la fidelización, relación que se observa pero no se demuestra causalmente.
5. El equipo de retención/CRM debería decidir mantener o escalar las acciones que coinciden con este crecimiento sostenido.


## Reto 2 — Diagnóstico sin confundir correlación con causalidad


In [16]:
# Mes con mayor caida porcentual de la North Star.
# Se excluye el mes de calentamiento y el primero comparado contra el.
peor = (kpi.filter((pl.col("mes") > MES_CALENTAMIENTO) & pl.col("var_ns_pct").is_not_null())
        .sort("var_ns_pct")
        .head(1))
print("Mayor caida mensual de la North Star")
display(peor.select(["mes", "var_ns_pct", "var_clientes_rec_pct", "var_frecuencia_pct",
                     "tasa_cancelacion_pct", "concentracion_top10_pct"]))


Mayor caida mensual de la North Star


mes,var_ns_pct,var_clientes_rec_pct,var_frecuencia_pct,tasa_cancelacion_pct,concentracion_top10_pct
str,f64,f64,f64,f64,f64
"""2011-06""",-9.441385,-4.449938,-5.223907,18.394845,29.2166


A partir de la salida anterior, escriba una conclusión en tres capas:

- **Evidencia:** lo que muestran los datos.
- **Inferencia:** explicación plausible, sin afirmar causalidad.
- **Decisión:** acción concreta que debería evaluar el responsable.

**Respuesta:**
- **Evidencia:** en junio de 2011 la North Star cayó 9,4 %, con caídas simultáneas en clientes recurrentes (-4,4 %) y frecuencia (-5,2 %), mientras la concentración Top 10 subió a 29,2 %, el segundo valor más alto del periodo.
- **Inferencia:** la caída conjunta en ambos drivers sugiere una posible pausa estacional de mitad de año o menor esfuerzo comercial, sin que los datos permitan afirmar una causa específica.
- **Decisión:** el responsable de retención debería revisar qué ocurrió operativamente en junio (campañas, catálogo, incidencias de servicio) antes de fijar metas para el próximo trimestre.


## Diccionario de KPI — completar

| KPI | Fórmula / unidad | Fuente / frecuencia | Responsable | Meta / alerta | Acción |
|---|---|---|---|---|---|
| Compras recurrentes/mes | Nº de facturas con n_compra_cliente ≥ 2 | UCI / mensual | Gerencia de retención | Crecimiento mensual positivo (supuesto académico) | Escalar o ajustar campañas de fidelización |
| Clientes recurrentes activos | Nº de CustomerID distintos con ≥ 2 compras | UCI / mensual | CRM | Crecimiento sostenido mes a mes (supuesto) | Priorizar segmentos con caída |
| Frecuencia recurrente | compras_recurrentes / clientes_recurrentes | UCI / mensual | Producto / CRM | ≥ 1.5 compras por cliente (supuesto) | Activar programas de reactivación si cae |
| Tasa de cancelación | facturas_canceladas / facturas_total × 100 | UCI / mensual | Operaciones / Calidad | < 18 % (supuesto, cercano al promedio del periodo) | Revisar causas de devolución si supera el umbral |
| Concentración Top 10 | ingreso Top 10 clientes / ingreso total × 100 | UCI / mensual | Finanzas / Ventas | < 25 % (supuesto) | Diversificar cartera de clientes si se supera |

> Las metas y alertas propuestas son supuestos académicos de gestión, no metas oficiales de la empresa.


---

## Reto de aplicación y retroalimentación

En esta sección se aplicarán los procedimientos desarrollados durante la sesión a nuevas situaciones de análisis. Cada ejercicio requiere modificar, completar o construir código a partir de las tablas ya procesadas. Posteriormente, los resultados obtenidos deberán interpretarse brevemente desde una perspectiva empresarial. El propósito es comprobar la comprensión de las técnicas utilizadas y fortalecer la capacidad de adaptar el análisis ante nuevas preguntas de negocio.

**Indicaciones generales.** Los ejercicios operan sobre `kpi_reto`, copia de trabajo de la tabla de KPI, y sobre `facturas`, ya construida en la Actividad 2. Los datos proceden íntegramente del repositorio UCI; no corresponde generar ni sustituir valores en ningún caso. Cada respuesta escrita no debe exceder cuatro líneas.

**Duración en sesión:** 25 minutos. Los ejercicios que no concluyan se completan como avance del entregable.


In [17]:
# Copia de trabajo para la sección de retos.
kpi_reto = kpi.clone()
print("Copia de trabajo creada:", kpi_reto.shape)
print("Columnas disponibles:", kpi_reto.columns)
display(kpi_reto.head())


Copia de trabajo creada: (11, 16)
Columnas disponibles: ['mes', 'facturas_validas', 'clientes_activos', 'ingresos', 'compras_recurrentes', 'clientes_recurrentes', 'ingresos_recurrentes', 'frecuencia_recurrente', 'participacion_ingreso_recurrente_pct', 'ns_reconstruida', 'error_reconstruccion', 'tasa_cancelacion_pct', 'concentracion_top10_pct', 'var_ns_pct', 'var_clientes_rec_pct', 'var_frecuencia_pct']


mes,facturas_validas,clientes_activos,ingresos,compras_recurrentes,clientes_recurrentes,ingresos_recurrentes,frecuencia_recurrente,participacion_ingreso_recurrente_pct,ns_reconstruida,error_reconstruccion,tasa_cancelacion_pct,concentracion_top10_pct,var_ns_pct,var_clientes_rec_pct,var_frecuencia_pct
str,u32,u32,f64,u32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2011-01""",987,741,569445.04,570,370,296614.01,1.540541,52.088259,570.0,0.0,20.145631,33.944082,null,null,null
"""2011-02""",997,758,447137.35,617,412,299937.36,1.497573,67.079469,617.0,0.0,16.971714,20.054254,8.245614,11.351351,-2.789133
"""2011-03""",1321,974,595500.76,869,563,411030.13,1.543517,69.022604,869.0,0.0,18.406424,19.073929,40.842788,36.650485,3.067901
"""2011-04""",1149,856,469200.361,849,587,357273.97,1.446337,76.145289,849.0,1.1369e-13,16.979769,15.638458,-2.301496,4.262877,-6.295983
"""2011-05""",1555,1056,678594.56,1271,809,566039.71,1.571075,83.413535,1271.0,0.0,15.900487,18.880129,49.705536,37.819421,8.624412


### Ejercicio 1 — Construcción de una tasa de recurrencia mensual

La North Star del laboratorio se expresa en cantidad de compras recurrentes, magnitud absoluta que crece cuando aumenta la actividad total del negocio. Una magnitud absoluta, sin embargo, no permite distinguir si la recurrencia mejora o si simplemente hay más transacciones de cualquier tipo.

La tasa de recurrencia corrige esa limitación: expresa qué proporción de las facturas del mes corresponde a clientes que ya habían comprado con anterioridad. Se trata de una magnitud relativa y, por tanto, comparable entre meses de distinto volumen.

**Se solicita:**

1. Calcular, para cada mes, la cantidad total de facturas y la cantidad de facturas recurrentes a partir de `facturas`.
2. Construir la tasa de recurrencia mensual expresada en porcentaje e incorporarla a `kpi_reto`.
3. Identificar el mes con la tasa más alta y el mes con la tasa más baja.
4. Comparar el ordenamiento por tasa de recurrencia con el ordenamiento por compras recurrentes.

**Tiempo estimado:** 5 minutos.


In [18]:
# ---------------------------------------------------------------------------
# CELDA DE TRABAJO — Ejercicio 1
# ---------------------------------------------------------------------------
recu_mensual = (facturas.group_by("mes")
    .agg([
        pl.col("InvoiceNo").n_unique().alias("facturas_totales"),
        pl.col("es_recurrente").sum().alias("facturas_recurrentes"),
    ])
    .with_columns(
        (100 * pl.col("facturas_recurrentes") / pl.col("facturas_totales"))
        .alias("tasa_recurrencia_pct"))
    .sort("mes"))

kpi_reto = kpi_reto.join(
    recu_mensual.select(["mes", "tasa_recurrencia_pct"]), on="mes", how="left")

print("Ordenado por tasa de recurrencia (desc):")
display(kpi_reto.select(["mes", "compras_recurrentes", "tasa_recurrencia_pct"])
        .sort("tasa_recurrencia_pct", descending=True))

print("Ordenado por compras recurrentes (desc):")
display(kpi_reto.select(["mes", "compras_recurrentes", "tasa_recurrencia_pct"])
        .sort("compras_recurrentes", descending=True))


Ordenado por tasa de recurrencia (desc):


mes,compras_recurrentes,tasa_recurrencia_pct
str,u32,f64
"""2011-11""",2334,87.843432
"""2011-08""",1111,86.796875
"""2011-07""",1143,85.875282
"""2011-09""",1456,82.962963
"""2011-06""",1151,82.627423
"""2011-05""",1271,81.736334
"""2011-10""",1571,81.441161
"""2011-04""",849,73.890339
"""2011-03""",869,65.783497


Ordenado por compras recurrentes (desc):


mes,compras_recurrentes,tasa_recurrencia_pct
str,u32,f64
"""2011-11""",2334,87.843432
"""2011-10""",1571,81.441161
"""2011-09""",1456,82.962963
"""2011-05""",1271,81.736334
"""2011-06""",1151,82.627423
"""2011-07""",1143,85.875282
"""2011-08""",1111,86.796875
"""2011-03""",869,65.783497
"""2011-04""",849,73.890339


**Pregunta 3.** ¿Coincide el mes con mayor cantidad de compras recurrentes con el mes de mayor tasa de recurrencia? Explique qué implicancia tiene esa diferencia para la medición del negocio.

**Respuesta (máximo cuatro líneas):**

---
No coinciden del todo: noviembre tiene tanto la mayor cantidad (2334) como la mayor tasa (87,8 %), pero el menor volumen absoluto corresponde a enero (570, mes de calentamiento), mientras que —excluyéndolo— la menor tasa es febrero (61,9 %) y el menor volumen es abril (849). Crecer en compras recurrentes no siempre implica que la recurrencia mejore en proporción: puede deberse solo a mayor actividad total del negocio.


### Ejercicio 2 — Incorporación del ticket promedio como driver económico

El árbol de métricas descompuso la North Star en clientes recurrentes y frecuencia de compra. Ninguno de esos dos drivers recoge el valor económico de cada transacción: dos meses con idéntica cantidad de compras recurrentes pueden presentar ingresos muy distintos si el importe promedio de la factura cambia.

El ticket promedio de la factura recurrente completa esa descripción y permite establecer si el crecimiento observado proviene de mayor actividad o de mayor valor por transacción.

**Se solicita:**

1. Calcular, a partir de `facturas`, el importe promedio de las facturas recurrentes de cada mes.
2. Incorporar el resultado a `kpi_reto` en la columna `ticket_promedio_recurrente`.
3. Calcular su variación mensual porcentual, siguiendo el mismo procedimiento empleado para los demás drivers.
4. Contrastar los meses de mayor caída de la North Star con el comportamiento del ticket promedio en esos mismos meses.

**Tiempo estimado:** 5 minutos.


In [19]:
# ---------------------------------------------------------------------------
# CELDA DE TRABAJO — Ejercicio 2
# ---------------------------------------------------------------------------
ticket = (facturas.filter(pl.col("es_recurrente"))
    .group_by("mes")
    .agg(pl.col("importe_factura").mean().alias("ticket_promedio_recurrente"))
    .sort("mes"))

kpi_reto = kpi_reto.join(ticket, on="mes", how="left").sort("mes")
kpi_reto = kpi_reto.with_columns(
    (100 * (pl.col("ticket_promedio_recurrente")
            / pl.col("ticket_promedio_recurrente").shift(1) - 1))
    .alias("var_ticket_pct"))

display(kpi_reto.select(["mes", "ticket_promedio_recurrente", "var_ticket_pct", "var_ns_pct"]))


mes,ticket_promedio_recurrente,var_ticket_pct,var_ns_pct
str,f64,f64,f64
"""2011-01""",520.375456,null,null
"""2011-02""",486.122139,-6.582424,8.245614
"""2011-03""",472.992094,-2.700977,40.842788
"""2011-04""",420.817397,-11.030776,-2.301496
"""2011-05""",445.34989,5.829724,49.705536
"""2011-06""",495.34318,11.225621,-9.441385
"""2011-07""",465.507157,-6.023304,-0.695048
"""2011-08""",511.488461,9.877679,-2.79965
"""2011-09""",554.041532,8.319459,31.053105


**Pregunta 4.** En el mes de mayor caída de la North Star, ¿el ticket promedio recurrente aumentó o disminuyó? ¿Qué sugiere ese comportamiento conjunto?

**Respuesta (máximo cuatro líneas):**

---
En junio de 2011, mes de mayor caída de la North Star (-9,4 %), el ticket promedio recurrente en realidad aumentó (+11,2 % respecto a mayo, de 445 a 495 GBP). Esto sugiere que la caída no se debió a compras de menor valor, sino a menos clientes y transacciones recurrentes: quienes compraron gastaron más, pero fueron menos.


### Ejercicio 3 — Definición de un guardrail de dependencia del cliente principal

La Actividad 2 incorporó un guardrail de concentración sobre los diez principales clientes. Ese umbral describe la dependencia agregada del negocio, pero no revela si dicha concentración se explica por un único cliente de gran tamaño, situación que constituye un riesgo operativo de naturaleza distinta.

Un guardrail de dependencia del cliente principal mide qué proporción del ingreso mensual corresponde al cliente de mayor facturación. Su finalidad no es optimizarse, sino advertir cuando la concentración alcanza un nivel que compromete la continuidad del negocio.

**Se solicita:**

1. Calcular, para cada mes, el ingreso del cliente de mayor facturación. La tabla `ordenado` ya dispone de la columna `ranking` calculada dentro de cada mes.
2. Expresar ese ingreso como porcentaje del ingreso total del mes e incorporarlo a `kpi_reto`.
3. Declarar explícitamente un umbral de alerta como criterio pedagógico del ejercicio, no como norma sectorial.
4. Identificar los meses que superan dicho umbral.

**Tiempo estimado:** 5 minutos.


In [20]:
# ---------------------------------------------------------------------------
# CELDA DE TRABAJO — Ejercicio 3
# ---------------------------------------------------------------------------
top1 = (ordenado.filter(pl.col("ranking") == 1)
        .select(["mes", "ingreso_cliente"])
        .rename({"ingreso_cliente": "ingreso_top1"}))

kpi_reto = kpi_reto.join(top1, on="mes", how="left")
kpi_reto = kpi_reto.with_columns(
    (100 * pl.col("ingreso_top1") / pl.col("ingresos")).alias("dependencia_top1_pct"))

# Umbral de alerta declarado como supuesto pedagogico, no como norma sectorial.
UMBRAL_DEPENDENCIA_TOP1 = 8.0

display(kpi_reto.select(["mes", "ingreso_top1", "ingresos", "dependencia_top1_pct"])
        .sort("dependencia_top1_pct", descending=True))

alerta = kpi_reto.filter(pl.col("dependencia_top1_pct") > UMBRAL_DEPENDENCIA_TOP1)
print(f"Meses que superan el umbral de {UMBRAL_DEPENDENCIA_TOP1}%:", alerta["mes"].to_list())


mes,ingreso_top1,ingresos,dependencia_top1_pct
str,f64,f64,f64
"""2011-01""",77183.6,569445.04,13.554179
"""2011-09""",75412.64,952838.382,7.914526
"""2011-06""",41959.44,661213.69,6.345821
"""2011-08""",40327.81,645343.9,6.249042
"""2011-02""",22797.46,447137.35,5.098536
"""2011-10""",52681.27,1.0393e6,5.068827
"""2011-04""",21535.9,469200.361,4.589915
"""2011-07""",26464.99,600091.011,4.410163
"""2011-05""",28408.14,678594.56,4.18632


Meses que superan el umbral de 8.0%: ['2011-01']


**Pregunta 5.** ¿En cuántos meses el cliente principal supera el umbral declarado y qué decisión empresarial justificaría ese resultado?

**Respuesta (máximo cuatro líneas):**

---
Con el umbral académico declarado (8 % del ingreso mensual concentrado en un solo cliente), solo 1 de 11 meses (enero, con 13,6 %) lo supera. Como enero también concentra la mayor cancelación y la mayor concentración Top 10, la decisión razonable sería reforzar la diversificación de cartera al inicio del año, sin alarmarse por el resto del periodo.


### Ejercicio 4 — Consulta de meses saludables con condiciones múltiples en DuckDB

En la Actividad 2 se empleó DuckDB para ordenar los meses según la variación de la North Star. Una consulta orientada a la decisión, sin embargo, no se limita a ordenar: delimita el subconjunto de periodos que satisfacen simultáneamente el objetivo de crecimiento y las restricciones fijadas por los guardrails.

Un mes de crecimiento acompañado de un deterioro en la tasa de cancelación no constituye un mes saludable. Esta consulta materializa esa distinción en una regla reproducible.

**Se solicita:**

1. Construir sobre `kpi_reto` una consulta SQL que incluya `SELECT`, `WHERE`, condiciones múltiples enlazadas con `AND` y `ORDER BY`.
2. Establecer como criterios una variación de la North Star positiva y una tasa de cancelación inferior al promedio del periodo.
3. Ordenar el resultado por variación de la North Star de mayor a menor.
4. Determinar cuántos meses satisfacen ambos criterios.

**Tiempo estimado:** 5 minutos.


In [21]:
# ---------------------------------------------------------------------------
# CELDA DE TRABAJO — Ejercicio 4
# ---------------------------------------------------------------------------
saludables = duckdb.sql("""
    SELECT mes, var_ns_pct, tasa_cancelacion_pct
    FROM kpi_reto
    WHERE var_ns_pct > 0
      AND tasa_cancelacion_pct < (SELECT AVG(tasa_cancelacion_pct) FROM kpi_reto)
    ORDER BY var_ns_pct DESC
""").pl()

display(saludables)
print("Cantidad de meses saludables:", saludables.height)


mes,var_ns_pct,tasa_cancelacion_pct
str,f64,f64
"""2011-05""",49.705536,15.900487
"""2011-11""",48.567791,13.869086
"""2011-09""",31.053105,15.495669
"""2011-10""",7.898352,14.759169


Cantidad de meses saludables: 4


**Pregunta 6.** ¿Cuántos meses del periodo pueden calificarse como saludables según los criterios establecidos y qué sugiere esa proporción sobre la solidez del crecimiento?

**Respuesta (máximo cuatro líneas):**

---
Solo 4 de los 11 meses (mayo, septiembre, octubre y noviembre) califican como saludables, apenas el 36 % del periodo. Esa proporción sugiere que el crecimiento de la North Star no es sólido de forma sostenida: en la mayoría de los meses, o la North Star no creció, o creció mientras la cancelación empeoraba, lo que exige monitoreo continuo antes de anunciar una tendencia consolidada.


### Ejercicio 5 — Representación de la relación entre el driver y la North Star

El tablero construido en la Actividad 2 presenta la North Star y su driver como series temporales paralelas. Esa disposición permite observar la evolución de ambas magnitudes, pero no muestra con claridad si sus variaciones se corresponden entre sí.

Un gráfico de dispersión que enfrente la variación del driver con la variación de la North Star hace visible esa correspondencia: los puntos alineados sobre una tendencia ascendente indican que el driver acompaña el movimiento del resultado, mientras que los puntos dispersos advierten que otros factores intervienen. Corresponde recordar que la correspondencia observada describe una asociación y no acredita una relación causal.

**Se solicita:**

1. Construir con Plotly un gráfico de dispersión que sitúe la variación porcentual de clientes recurrentes en el eje horizontal y la variación porcentual de la North Star en el eje vertical.
2. Identificar cada punto con el mes correspondiente.
3. Rotular los ejes y titular el gráfico de modo que resulte interpretable sin recurrir al código.
4. Excluir de manera explícita el mes de calentamiento, que carece de variación calculable.

**Tiempo estimado:** 5 minutos.


In [22]:
# ---------------------------------------------------------------------------
# CELDA DE TRABAJO — Ejercicio 5
# ---------------------------------------------------------------------------
dispersion = kpi_reto.filter(pl.col("mes") != MES_CALENTAMIENTO)

fig5 = px.scatter(dispersion.to_pandas(), x="var_clientes_rec_pct", y="var_ns_pct",
                   text="mes",
                   title="Variacion de clientes recurrentes vs. variacion de la North Star (2011)")
fig5.update_traces(textposition="top center")
fig5.update_layout(xaxis_title="Variacion de clientes recurrentes (%)",
                    yaxis_title="Variacion de la North Star (%)")
fig5.show()


**Pregunta 7.** ¿La variación del driver acompaña la variación de la North Star? Indique un mes que se aparte de esa correspondencia y proponga una explicación verificable.

**Respuesta (máximo cuatro líneas):**

---
En general sí acompaña: ambas variaciones suben o bajan juntas la mayoría de los meses (marzo, mayo, noviembre). Abril se aparta de esa correspondencia: los clientes recurrentes crecieron (+4,3 %) mientras la North Star cayó (-2,3 %). Una explicación verificable es que la frecuencia recurrente cayó fuertemente ese mes (-6,3 %), compensando el aumento de clientes.


## Informe ejecutivo breve

1. **Objetivo estratégico:** incrementar el valor recurrente del negocio sin deteriorar la calidad del servicio ni concentrar el riesgo en pocos clientes.
2. **North Star y justificación:** compras válidas de clientes recurrentes por mes; refleja valor repetido para el cliente, está ligada al ingreso (hasta 90 % de participación en noviembre), es accionable por retención y se descompone en drivers medibles.
3. **Hallazgo cuantitativo 1:** la North Star creció de 570 (enero) a 2334 (noviembre) compras recurrentes, con una caída puntual de -9,4 % en junio.
4. **Hallazgo cuantitativo 2:** la tasa de recurrencia mensual pasó de 57,8 % a 87,8 %, mostrando que mejoró la calidad de la base de clientes, no solo su volumen.
5. **Hallazgo cuantitativo 3:** solo 4 de 11 meses cumplen a la vez crecimiento de la North Star y cancelación por debajo del promedio; el crecimiento no es uniformemente saludable.
6. **Driver prioritario:** clientes recurrentes activos, ya que su variación explica la mayor parte del movimiento de la North Star en casi todos los meses.
7. **Guardrails:** tasa de cancelación (bajó de 20,1 % a 13,9 %) y concentración Top 10 (bajó de 33,9 % a 15,7 %); ambos mejoraron en el periodo.
8. **Decisión empresarial recomendada:** mantener e intensificar las campañas de fidelización asociadas al crecimiento de clientes recurrentes, e investigar puntualmente la caída de junio antes de fijar metas para el próximo trimestre.
9. **Limitación del dataset:** cubre solo un año (dic-2010 a dic-2011) de una sola empresa del Reino Unido, sin datos de costos ni de marketing, por lo que no permite generalizar a otros mercados ni establecer causalidad.


## Ticket de salida

Elija una métrica de su proyecto integrador. Explique por qué es accionable y qué decisión cambiaría si disminuyera 20 %.

**Respuesta:** En DiabeTech AI, la métrica elegida es la sensibilidad (recall) del clasificador de riesgo de hospitalización. Es accionable porque determina a cuántos pacientes con diabetes realmente en riesgo el sistema logra identificar para priorizar seguimiento clínico. Si el recall cayera 20 %, cambiaría la decisión de ajustar el umbral del scoring de riesgo (aceptando más falsos positivos a cambio de detectar más casos reales) o de reentrenar el modelo con variables adicionales antes de habilitarlo en el dashboard, ya que dejar de detectar pacientes de alto riesgo tiene un costo clínico directo.


## Referencias

- Chen, D. (2015). *Online Retail* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33  
- Croll, A., & Yoskovitz, B. (2013). *Lean Analytics*. O’Reilly Media.  
- Parmenter, D. (2020). *Key Performance Indicators* (4th ed.). Wiley.  
- Sharda, R., Delen, D., & Turban, E. (2024). *Business Intelligence, Analytics, Data Science, and AI* (5th ed.). Pearson.
